# RAG experiments
The Python modules in `src/angebotfinder/` contain the reusable functions. This notebook loads models once and calls those functions.
Run model loading once. After changing settings, rerun the pipeline configuration cell.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from angebotfinder.models import load_models, RERANKER_MODEL, GENERATION_MODEL
from angebotfinder.retrieval import load_retrieval_data, hybrid_search, rerank_search
from angebotfinder.generation import (
    prepare_context, build_messages, prepare_budgeted_messages,
    generate_answer, rewrite_question,
)
from angebotfinder.rag_pipeline import RAGPipeline

USE_REWRITING = True
MIN_RERANK_SCORE = 0


In [ ]:
offers, chunks, config, embeddings = load_retrieval_data(PROJECT_ROOT)
embedding_model, reranker_model, generator_model, generator_tokenizer = load_models(
    config["embedding_model"], RERANKER_MODEL, GENERATION_MODEL,
    local_files_only=True,
)
print(len(chunks), embeddings.shape)


In [ ]:
pipeline = RAGPipeline(
    embeddings,
    chunks,
    embedding_model,
    reranker_model,
    generator_model,
    generator_tokenizer,
    use_rewriting=USE_REWRITING,
    min_score=MIN_RERANK_SCORE,
)


In [ ]:
# question = "was kostet starbuks kaffeemischgetränk bei kauf land?"
question = "Tell me Newton's second law."
history = []

response = pipeline.ask_question(
    question,
    history=history,
)

print("Original:", response["question"])
print("Search:", response["search_question"])
print("Answer:", response["answer"])

history.append({
    "question": question,
    "answer": response["answer"],
})


## Inspect relevance scores
Compare relevant and unrelated questions before setting `MIN_RERANK_SCORE` near the top. 
`None` disables threshold filtering. A numeric value keeps results with scores at or above it. 
Use several questions to choose the cutoff; these two are a starting check.


In [ ]:
test_questions = [
    "Was kostet STARBUCKS Kaffeemischgetränk bei Kaufland?",
    "Tell me Newton's second law.",
]

for test_question in test_questions:
    test_search_question = test_question
    if USE_REWRITING:
        test_search_question = rewrite_question(
            test_question, generator_model, generator_tokenizer
        )

    test_results = rerank_search(
        test_search_question,
        embeddings,
        chunks,
        embedding_model=embedding_model,
        reranker=reranker_model,
        bm25=pipeline.bm25,
    )

    print("\nQuestion:", test_question)
    print("Search:", test_search_question)
    for result in test_results:
        print(round(result["reranker_score"], 3), result["text"])


In [ ]:
if MIN_RERANK_SCORE is None:
    print("Choose MIN_RERANK_SCORE from the inspected scores, then rerun this cell.")
else:
    pipeline.min_score = MIN_RERANK_SCORE
    response = pipeline.ask_question(
        # "is there any offer for nescaffe gold?",
        # "Was kostet STARBUCKS Kaffeemischgetränk bei Kaufland?",
        "Which stores have Haribo offers? Show the listed price and pack size at each store.",
        top_k=5,
        max_new_tokens=300,
    )
    print(response["status"])
    print(response["question"])
    print(response["search_question"])
    print(response["answer"])
